# Deep MLP
In this file i'm going to implement a deep neural net, with enforcment on hierarchy for the classes.

In [122]:
import numpy as np
import pandas as pd
from collections import defaultdict
import torch.optim as optim

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader, TensorDataset

from skfp.fingerprints import ECFPFingerprint, MACCSFingerprint, TopologicalTorsionFingerprint
from skfp.preprocessing import MolFromSmilesTransformer
from skfp.preprocessing import MolFromSmilesTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [5]:
test_sample = pd.read_parquet("tasks-2026/task1/chebi_dataset_test_empty.parquet")

In [ ]:
DATA_DIR = "tasks-2026/task1"

df_train = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_train.parquet")
df_test = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_test_empty.parquet")

label_cols = [c for c in df_train.columns if c.startswith("class_")]
num_classes = len(label_cols)
mols = df_test.iloc[:,0:2].reset_index(drop=True)

print(f"Train: {len(df_train)} molecules, {num_classes} classes")
print(f"Test:  {len(df_test)} molecules")

Train: 33668 molecules, 500 classes
Test:  11223 molecules


In [ ]:
all_smiles = pd.concat([df_train["SMILES"], df_test["SMILES"]], ignore_index=True).tolist()

mol_transformer = MolFromSmilesTransformer()
all_mols = mol_transformer.transform(all_smiles)

ecfp = ECFPFingerprint(fp_size=2048, radius=2, n_jobs=-1)
maccs = MACCSFingerprint(n_jobs=-1)
torsion = TopologicalTorsionFingerprint(fp_size=2048, n_jobs=-1)

print("Computing ECFP...")
fp_ecfp = ecfp.transform(all_mols)
print("Computing MACCS...")
fp_maccs = maccs.transform(all_mols)
print("Computing TopologicalTorsion...")
fp_torsion = torsion.transform(all_mols)

X_all = np.hstack([fp_ecfp, fp_maccs, fp_torsion]).astype(np.float32)
print(f"Total fingerprint dim: {X_all.shape[1]}")

n_train = len(df_train)
X_train_full = X_all[:n_train]
X_test = X_all[n_train:]

Y_train_full = df_train[label_cols].values.astype(np.float32)

print(f"X_train: {X_train_full.shape}, X_test: {X_test.shape}")

In [ ]:
def parse_obo(path):
    child_to_parents = defaultdict(list)
    current_id = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("id: class_"):
                current_id = line.split("id: ")[1]
            elif line.startswith("is_a: class_"):
                parent = line.split("is_a: ")[1].strip()
                child_to_parents[current_id].append(parent)
    return child_to_parents

child_to_parents = parse_obo(f"{DATA_DIR}/chebi_classes.obo")

# Load class names for readable reports
class_defs = pd.read_csv(f"{DATA_DIR}/chebi_class_definitions.csv")
class_id_to_name = dict(zip(class_defs["chebi_id"], class_defs["name"]))

class_to_idx = {c: i for i, c in enumerate(label_cols)}

# parent_mask[i, j] = 1 if class j is a direct parent of class i
parent_mask = torch.zeros(num_classes, num_classes)
for child, parents in child_to_parents.items():
    if child in class_to_idx:
        for p in parents:
            if p in class_to_idx:
                parent_mask[class_to_idx[child], class_to_idx[p]] = 1.0

parent_mask_np = parent_mask.numpy()
print(f"Hierarchy: {int(parent_mask.sum())} direct parent-child edges")

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_full, Y_train_full, test_size=0.25, random_state=42
)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val))
test_ds = TensorDataset(torch.from_numpy(X_test))

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


INPUT_DIM = X_train_full.shape[1]
HIDDEN_DIM_1 = 512
# HIDDEN_DIM_2 = 1024
# HIDDEN_DIM_3 = 512
DROPOUT = 0.3

model = MLP(INPUT_DIM, HIDDEN_DIM_1, num_classes, DROPOUT).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Input dim: {INPUT_DIM}")
print(f"Model parameters: {total_params:,}")

In [70]:
class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, num_classes, dropout=0.3):
        super().__init__()
        
        layers = []
        current_dim = input_dim
        
        # Tworzymy kolejne warstwy na podstawie listy hidden_layers
        for h_dim in hidden_layers:
            layers.append(nn.Linear(current_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.GELU()) # GELU jest lepsze dla głębokich sieci niż ReLU
            layers.append(nn.Dropout(dropout))
            current_dim = h_dim
            
        self.feature_extractor = nn.Sequential(*layers)
        
        # Ostatnia warstwa klasyfikująca
        self.classifier = nn.Linear(current_dim, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        return self.classifier(x)

In [137]:
HIDDEN_LAYERS = [2024,1012, 512] # Możesz tu wpisać ile chcesz warstw!
model = DeepMLP(INPUT_DIM, HIDDEN_LAYERS, num_classes, DROPOUT).to(device)

In [ ]:
pos_freq = Y_train_full.mean(axis=0)
pos_freq = np.clip(pos_freq, 0.001, 0.999)
pos_weight = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float).to(device)
pos_weight = torch.clamp(pos_weight, max=20.0)

# criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# print(f"Pos weight range: [{pos_weight.min():.2f}, {pos_weight.max():.2f}]")

In [115]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, pos_weight=None):
        super(FocalLoss, self). __init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none', pos_weight=self.pos_weight)
        pt = torch.exp(-bce_loss) # prawdopodobieństwo trafienia
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()
    
criterion = FocalLoss(pos_weight=pos_weight, gamma=2.0)

In [ ]:
def hierarchical_loss(logits, parent_mask):

    children_idx, parents_idx = torch.where(parent_mask == 1)
 
    log_child = logits[:, children_idx]
    log_parent = logits[:, parents_idx]
    
    violation = F.relu(log_child - log_parent)
    
    return violation.mean()

In [ ]:
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            x, y = batch[0].to(device), batch[1].to(device)
            logits = model(x)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.cpu().numpy())
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    all_preds = (all_probs >= threshold).astype(float)

    f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return f1_micro, f1_macro

In [ ]:
def evaluate(model, loader, parent_mask, threshold=0.6):
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            x = batch[0].to(device)
            logits = model(x)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            
            if len(batch) > 1:
                all_labels.append(batch[1].cpu().numpy())
    
    all_probs = np.vstack(all_probs)

    children_idx, parents_idx = np.where(parent_mask.cpu().numpy() == 1)
    

    p_children = all_probs[:, children_idx]
    p_parents = all_probs[:, parents_idx]
    
    violations = p_children > p_parents
    total_violations = np.sum(violations)

    violation_rate = total_violations / (all_probs.shape[0] * len(children_idx))


    if len(all_labels) > 0:
        all_labels = np.vstack(all_labels)
        all_preds = (all_probs >= threshold).astype(float)
        f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)
        f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        return f1_micro, f1_macro, violation_rate, total_violations
    
    return f1_micro, f1_macro, violation_rate, total_violations

In [ ]:
LR = 1e-3
EPOCHS = 40
PATIENCE = 10

# optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
# scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=40, T_mult=2, eta_min=1e-6)

# AdamW z nieco wyższym weight_decay dla lepszej generalizacji
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Scheduler - wróćmy do T_0=40, skoro to był Twój rekord
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=40, T_mult=1, eta_min=1e-6
)

# scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, 
#     mode='max',       # 'max', bo chcemy zwiększać F1-score
#     factor=0.9,       # Zmniejsza LR o połowę (np. z 0.001 na 0.0005)
#     patience=1      # Czeka 3 epoki bez poprawy zanim zareaguje
# )

best_f1 = 0
patience_counter = 0
history = {"train_loss": [], "val_f1_micro": [], "val_f1_macro": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    n_samples = 0
    for batch in train_loader:
        x, y = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        h_loss = hierarchical_loss(logits, parent_mask)
        t_loss = loss + 0.7 * h_loss
        t_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n_samples += x.size(0)

    f1_micro, f1_macro, violation_rate, total_violations = evaluate(model, val_loader, parent_mask)
    scheduler.step()
    avg_loss = total_loss / n_samples


    history["train_loss"].append(avg_loss)
    history["val_f1_micro"].append(f1_micro)
    history["val_f1_macro"].append(f1_macro)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | "
          f"F1-micro: {f1_micro:.4f} | F1-macro: {f1_macro:.4f} | LR: {current_lr:.2e} | Violation_rate: {violation_rate:.2%}")

    


Epoch 01 | Loss: 0.1635 | F1-micro: 0.7544 | F1-macro: 0.5249 | LR: 9.98e-04 | Violation_rate: 2.62%
Epoch 02 | Loss: 0.0804 | F1-micro: 0.7857 | F1-macro: 0.5755 | LR: 9.94e-04 | Violation_rate: 2.02%
Epoch 03 | Loss: 0.0628 | F1-micro: 0.8181 | F1-macro: 0.6159 | LR: 9.86e-04 | Violation_rate: 1.77%
Epoch 04 | Loss: 0.0532 | F1-micro: 0.8331 | F1-macro: 0.6319 | LR: 9.76e-04 | Violation_rate: 1.63%
Epoch 05 | Loss: 0.0472 | F1-micro: 0.8426 | F1-macro: 0.6551 | LR: 9.62e-04 | Violation_rate: 1.54%
Epoch 06 | Loss: 0.0428 | F1-micro: 0.8593 | F1-macro: 0.6745 | LR: 9.46e-04 | Violation_rate: 1.50%
Epoch 07 | Loss: 0.0397 | F1-micro: 0.8614 | F1-macro: 0.6775 | LR: 9.26e-04 | Violation_rate: 1.45%
Epoch 08 | Loss: 0.0370 | F1-micro: 0.8718 | F1-macro: 0.6927 | LR: 9.05e-04 | Violation_rate: 1.34%
Epoch 09 | Loss: 0.0346 | F1-micro: 0.8756 | F1-macro: 0.6988 | LR: 8.80e-04 | Violation_rate: 1.30%
Epoch 10 | Loss: 0.0325 | F1-micro: 0.8746 | F1-macro: 0.6970 | LR: 8.54e-04 | Violation_ra

In [ ]:
def find_optimal_thresholds(model, loader, device, num_classes):
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            x, y = batch[0].to(device), batch[1].to(device)
            probs = torch.sigmoid(model(x))
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    
    best_thresholds = np.ones(num_classes) * 0.5
    
    for i in range(num_classes):
        best_f1 = -1

        thresholds = np.linspace(0.01, 0.99, 100)
        
        for t in thresholds:
            preds = (all_probs[:, i] >= t).astype(float)
            f1 = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresholds[i] = t
                
    return best_thresholds


optimal_thresholds = find_optimal_thresholds(model, val_loader, device, 500)

In [ ]:
def find_optimal_thresholds_fast(all_labels, all_probs, num_classes):
    best_thresholds = np.ones(num_classes) * 0.5
    
    for i in range(num_classes):
        precision, recall, thresholds = precision_recall_curve(all_labels[:, i], all_probs[:, i])
        

        f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
        
        if len(f1_scores) > 0:
            best_idx = np.argmax(f1_scores)

            if best_idx < len(thresholds):
                best_thresholds[i] = thresholds[best_idx]
                
    return best_thresholds

In [78]:
optimal_thresholds.size

500

In [ ]:
def final_predict(model, loader, thresholds, parent_mask, device):
    model.eval()
    all_final_preds = []
    

    if torch.is_tensor(parent_mask):
        p_mask_np = parent_mask.cpu().numpy()
    else:
        p_mask_np = parent_mask
        
    children_idx, parents_idx = np.where(p_mask_np == 1)
    

    thresholds = np.array(thresholds).reshape(1, -1) 

    with torch.no_grad():
        for batch in loader:

            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.to(device)
            
            logits = model(x)
            probs = torch.sigmoid(logits).cpu().numpy()
            

            preds = (probs >= thresholds).astype(float)
            

            for c, p in zip(children_idx, parents_idx):
                preds[:, p] = np.maximum(preds[:, p], preds[:, c])
            
            all_final_preds.append(preds)
            
    return np.vstack(all_final_preds)

In [80]:
y_pred = final_predict(model, test_loader, optimal_thresholds, parent_mask, device)

In [34]:
y_pred.shape

(11223, 500)

In [81]:
n_rows = len(df_test)
n_cols = len(label_cols)

pusta_macierz = np.zeros((n_rows, n_cols))
pusta_macierz.shape
df_klasy = pd.DataFrame(pusta_macierz, columns=label_cols, dtype='int8')
df_klasy.iloc[:, :] = y_pred
df_final = pd.concat([mols, df_klasy], axis=1)
df_final.to_parquet("final3.parquet")

In [ ]:
all_labels = []

with torch.no_grad():
    for batch in val_loader:

        labels = batch[1].reshape(-1, 500).cpu().numpy()
        all_labels.append(labels)

y_true_val = np.vstack(all_labels)

In [ ]:
f1_micro = f1_score(y_true_val, y_pred, average='micro')
f1_macro = f1_score(y_true_val, y_pred, average='macro')
print(f1_micro)
print(f1_macro)

In [47]:
y_prediction_final = final_predict(model, test_loader, optimal_thresholds, parent_mask, device)

In [ ]:
n_rows = len(df_test)
n_cols = len(label_cols)

pusta_macierz = np.zeros((n_rows, n_cols))
pusta_macierz.shape
df_klasy = pd.DataFrame(pusta_macierz, columns=label_cols, dtype='int8')
df_klasy.iloc[:, :] = y_prediction_final
df_final = pd.concat([mols, df_klasy], axis=1)
df_final.to_parquet("final.parquet")

In [48]:
y_prediction_final_pd = pd.DataFrame(y_prediction_final)
y_prediction_final_pd.to_parquet("submission.parquet")

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"])
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_f1_micro"], label="F1-micro")
axes[1].plot(history["val_f1_macro"], label="F1-macro")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves_mlp_fp.png", dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load("best_model_mlp_fp.pt", weights_only=True))
model.eval()

all_preds = []
with torch.no_grad():
    for batch in test_loader:
        x = batch[0].to(device)
        logits = model(x)
        preds = (torch.sigmoid(logits) >= 0.5).int()
        all_preds.append(preds.cpu().numpy())

all_preds = np.vstack(all_preds)

submission = pd.DataFrame(all_preds, columns=label_cols)
submission.insert(0, "mol_id", df_test["mol_id"].values)
submission.insert(1, "SMILES", df_test["SMILES"].values)

print(f"Submission shape: {submission.shape}")
submission.to_parquet("chebi_submission_mlp_fp.parquet", index=False)
print("Saved to chebi_submission_mlp_fp.parquet")